# V19 – Datenbanken Teil 1: Praxis

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- eine SQLite-Verbindung mit `sqlite3.connect(":memory:")` im `with`-Block öffnen,
- eine Tabelle mit `CREATE TABLE`, Einträge mit `INSERT`, Abfragen mit `SELECT` und Änderungen mit `UPDATE`/`DELETE` ausführen,
- Werte mit `?`-**Parameter-Binding** sicher übergeben,
- eine **CSV-Datei** mit `csv.DictReader` zeilenweise in eine SQLite-Tabelle überführen.

## ⏱️ Zeitbudget
Ca. 25 Minuten.

## 🧭 TL;DR
Wir arbeiten mit einer In-Memory-Datenbank, bauen Schritt für Schritt die `Produkte`-Tabelle auf, üben alle CRUD-Operationen und importieren am Ende die sechs Zeilen aus `testdaten/produkte.csv`. Drei Fill-in-Blanks bauen dir unterwegs Muskelgedächtnis auf.

## 📦 Voraussetzungen
- [01_theorie.ipynb](01_theorie.ipynb).


## 1. Verbindung aufbauen

Wir beginnen mit einer reinen **In-Memory-Datenbank**. Der Aufruf `sqlite3.connect(":memory:")` erzeugt eine leere Datenbank im RAM; mit `conn.cursor()` holen wir uns den Cursor, mit dem alle SQL-Befehle abgesetzt werden.


In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
print("Verbindung offen:", conn)
print("Cursor:", cur)


Verbindung offen: <sqlite3.Connection object at 0x00000127C6D0FF10>
Cursor: <sqlite3.Cursor object at 0x00000127C6F5EE40>


## 2. Tabelle anlegen

Wir erzeugen die Tabelle `Maschinen` mit ID, Name, Typ und einem Anschaffungsjahr. Der Primärschlüssel ist die `Maschinen_ID`.


In [2]:
cur.execute('''
CREATE TABLE Maschinen (
    Maschinen_ID INTEGER PRIMARY KEY,
    Name         TEXT    NOT NULL,
    Typ          TEXT    NOT NULL,
    Baujahr      INTEGER CHECK (Baujahr >= 1900)
)
''')
print("Tabelle Maschinen angelegt.")


Tabelle Maschinen angelegt.


### Zur Kontrolle: SQLite kennt `sqlite_master`

SQLite speichert sein Schema in einer internen Tabelle `sqlite_master`. Eine Abfrage gegen sie zeigt, welche Tabellen gerade existieren.


In [3]:
for row in cur.execute("SELECT name FROM sqlite_master WHERE type='table'"):
    print("Tabelle:", row[0])


Tabelle: Maschinen


## 3. INSERT mit Parameter-Binding

Jetzt fügen wir drei Maschinen ein. Wichtig: **immer** `?`-Platzhalter und Tupel als zweites Argument.


In [4]:
cur.execute("INSERT INTO Maschinen VALUES (?, ?, ?, ?)", (1, "Drehmaschine D-7", "Drehen", 2015))
cur.execute("INSERT INTO Maschinen VALUES (?, ?, ?, ?)", (2, "Fraesmaschine F-12", "Fraesen", 2019))
cur.execute("INSERT INTO Maschinen VALUES (?, ?, ?, ?)", (3, "3D-Drucker P-30", "Additiv", 2022))

print("Eingefuegte Zeilen insgesamt:", cur.execute("SELECT COUNT(*) FROM Maschinen").fetchone()[0])


Eingefuegte Zeilen insgesamt: 3


## 4. SELECT – lesen, filtern, sortieren

Die Grundform `SELECT ... FROM ...` liefert alle Zeilen. Mit `WHERE`, `ORDER BY` und `LIMIT` schränkst du das Ergebnis ein.


In [5]:
print("Alle Maschinen:")
for pid, name, typ, baujahr in cur.execute("SELECT * FROM Maschinen"):
    print(f"  #{pid} {name:<22} Typ={typ:<8} Baujahr={baujahr}")

print("\nNur Maschinen ab 2018, sortiert nach Baujahr:")
for name, bj in cur.execute("SELECT Name, Baujahr FROM Maschinen WHERE Baujahr >= ? ORDER BY Baujahr", (2018,)):
    print(f"  {bj} {name}")


Alle Maschinen:
  #1 Drehmaschine D-7       Typ=Drehen   Baujahr=2015
  #2 Fraesmaschine F-12     Typ=Fraesen  Baujahr=2019
  #3 3D-Drucker P-30        Typ=Additiv  Baujahr=2022

Nur Maschinen ab 2018, sortiert nach Baujahr:
  2019 Fraesmaschine F-12
  2022 3D-Drucker P-30


### Fill-in-Blank 1 — Älteste Maschine

Hol die **älteste** Maschine (kleinstes `Baujahr`) mit einer einzigen Query. Nutze `ORDER BY` und `LIMIT`. Speichere den Namen in `aeltester_name`.


In [6]:
aeltester_name = None
# TODO: Query absetzen, fetchone() verwenden, Name extrahieren


In [7]:
try:
    assert aeltester_name == "Drehmaschine D-7", f"Erwartet 'Drehmaschine D-7', bekommen: {aeltester_name!r}"
    print("✅ SELECT + ORDER BY + LIMIT sitzt.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `aeltester_name` existiert noch nicht.")


❌ Noch nicht ganz: Erwartet 'Drehmaschine D-7', bekommen: None


## 5. UPDATE – Werte ändern

Der 3D-Drucker wird umbenannt, weil die Firma einen neuen Naming-Standard einführt. Denk an die `WHERE`-Klausel!


In [8]:
cur.execute("UPDATE Maschinen SET Name = ? WHERE Maschinen_ID = ?", ("3D-Drucker PRO-30", 3))
for row in cur.execute("SELECT * FROM Maschinen WHERE Maschinen_ID = ?", (3,)):
    print("Nach UPDATE:", row)


Nach UPDATE: (3, '3D-Drucker PRO-30', 'Additiv', 2022)


## 6. DELETE – Zeilen löschen

Die alte Drehmaschine D-7 wird außer Dienst gestellt und aus der Tabelle entfernt.


In [9]:
cur.execute("DELETE FROM Maschinen WHERE Maschinen_ID = ?", (1,))
print("Verbleibende Zeilen:", cur.execute("SELECT COUNT(*) FROM Maschinen").fetchone()[0])
for row in cur.execute("SELECT * FROM Maschinen"):
    print(" ", row)


Verbleibende Zeilen: 2
  (2, 'Fraesmaschine F-12', 'Fraesen', 2019)
  (3, '3D-Drucker PRO-30', 'Additiv', 2022)


### Fill-in-Blank 2 — Sicheres INSERT

Füge eine neue Maschine mit `Maschinen_ID = 4`, Name `"Laserschneider L-5"`, Typ `"Laser"`, Baujahr `2024` ein. Nutze Parameter-Binding.


In [10]:
neuer_name = "Laserschneider L-5"
# TODO: cur.execute(...) mit Parameter-Binding aufrufen


In [11]:
try:
    row = cur.execute("SELECT Name, Typ, Baujahr FROM Maschinen WHERE Maschinen_ID = ?", (4,)).fetchone()
    assert row is not None, "Maschine 4 nicht gefunden - INSERT fehlt"
    assert row == ("Laserschneider L-5", "Laser", 2024), f"Falsche Werte: {row}"
    print("✅ INSERT mit Parameter-Binding korrekt.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")


❌ Noch nicht ganz: Maschine 4 nicht gefunden - INSERT fehlt


## 7. Warnung: Niemals String-Konkatenation!

Zur Einprägung zeigen wir noch einmal, wie **falsch** aussieht, was du nicht tun sollst. Der Code ist bewusst nur als String, er wird nicht ausgeführt.


In [12]:
boeser_code = '''
name = input("Name? ")
# UNSICHER - niemals so schreiben!
cur.execute(f"SELECT * FROM Maschinen WHERE Name = '{name}'")
'''
print(boeser_code)
print("-> bei Eingabe \"' OR '1'='1\" bekommt der Angreifer ALLE Zeilen.")



name = input("Name? ")
# UNSICHER - niemals so schreiben!
cur.execute(f"SELECT * FROM Maschinen WHERE Name = '{name}'")

-> bei Eingabe "' OR '1'='1" bekommt der Angreifer ALLE Zeilen.


## 8. CSV-Import – `produkte.csv` in SQLite

Jetzt kombinieren wir die Module `csv` und `sqlite3`. Wir legen eine neue Tabelle `Produkte` an und fügen zeilenweise alle sechs Produkte aus `testdaten/produkte.csv` ein.


In [13]:
import csv

cur.execute('''
CREATE TABLE Produkte (
    Produkt_ID               INTEGER PRIMARY KEY,
    Produktname              TEXT    NOT NULL,
    Produktionszeit_Minuten  INTEGER,
    Material_pro_Stueck_kg   REAL
)
''')

with open("testdaten/produkte.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for zeile in reader:
        cur.execute(
            "INSERT INTO Produkte VALUES (?, ?, ?, ?)",
            (
                int(zeile["Produkt_ID"]),
                zeile["Produktname"],
                int(zeile["Produktionszeit_Minuten"]),
                float(zeile["Material_pro_Stueck_kg"]),
            ),
        )

print("Importierte Produkte:", cur.execute("SELECT COUNT(*) FROM Produkte").fetchone()[0])
for row in cur.execute("SELECT * FROM Produkte ORDER BY Produkt_ID"):
    print(" ", row)


Importierte Produkte: 6
  (1, 'Zahnrad Z42', 25, 1.8)
  (2, 'Welle W-18', 35, 3.2)
  (3, 'Flansch F-90', 18, 2.5)
  (4, 'Buchse B-25', 12, 0.8)
  (5, 'Gehaeuse G-55', 45, 5.5)
  (6, 'Verbindungselement VE-12', 8, 0.3)


### Fill-in-Blank 3 — Filter-Query

Liste alle Produkte auf, deren **Material_pro_Stueck_kg** größer als `2.0` ist. Speichere das Ergebnis als Liste von Tupeln in `schwere_produkte` (nur Spalten `Produktname, Material_pro_Stueck_kg`).


In [14]:
schwere_produkte = []
# TODO: SELECT mit WHERE, fetchall()


In [15]:
try:
    erwartete_namen = {"Welle W-18", "Flansch F-90", "Gehaeuse G-55"}
    tatsaechliche_namen = {name for name, _masse in schwere_produkte}
    assert tatsaechliche_namen == erwartete_namen, (
        f"Erwartet {erwartete_namen}, bekommen: {tatsaechliche_namen}"
    )
    assert all(masse > 2.0 for _name, masse in schwere_produkte), "Alle Massen muessen > 2.0 sein"
    print("✅ Filter-Query korrekt.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Variable `schwere_produkte` existiert noch nicht.")


❌ Noch nicht ganz: Erwartet {'Welle W-18', 'Gehaeuse G-55', 'Flansch F-90'}, bekommen: set()


## 9. Aufräumen

Ein letztes `conn.close()` schließt die Verbindung. Bei In-Memory-Datenbanken ist das kosmetisch – die Daten sind ohnehin flüchtig.


In [16]:
conn.close()
print("Verbindung geschlossen.")


Verbindung geschlossen.


## ✅ Zusammenfassung
- `sqlite3.connect(":memory:")` liefert eine blitzschnelle Sandbox für Übungen.
- CREATE, INSERT, SELECT, UPDATE, DELETE decken das komplette CRUD ab.
- `?`-Parameter sind Pflicht – String-Konkatenation ist ein Sicherheits-Risiko.
- CSV-Import ist ein einfacher Dreier-Schritt: Tabelle anlegen → zeilenweise lesen → INSERT mit Typ-Konvertierung.

## ➡️ Nächster Schritt
Weiter mit [03_aufgaben.ipynb](03_aufgaben.ipynb) – dort übst du CRUD an fünf Aufgaben auf der `Produkte`-Tabelle.
